# Computational Exercises 1

## Exercise 1

We have the following basic code:

In [1]:
import firedrake as fd
from firedrake.output import VTKFile

# Finite element mesh
Nx, Ny = 32, 32
Lx, Ly = 1.0, 1.0
msh = fd.RectangleMesh(Nx, Ny, Lx, Ly, quadrilateral=False)

# Space of functions
Vd = fd.FunctionSpace(msh, "CG", degree=1)

# Test and Trial functions
u  = fd.TrialFunction(Vd)
v  = fd.TestFunction(Vd)

# Boundary conditions
u_boundary = fd.Constant(0.0)
bc = fd.DirichletBC(Vd, u_boundary, "on_boundary")

# Source term
f  = fd.Constant(1.0)

x  = fd.SpatialCoordinate(msh)
mu = fd.Constant(1.0) # this could be a function of x

# Bilinear form (lhs) and linear form (rhs)
a  = fd.inner(mu * fd.grad(u), fd.grad(v)) * fd.dx
L  = fd.inner(f, v) * fd.dx

# Solve the problem
ud = fd.Function(Vd)
opts={"ksp_type": "preonly", "pc_type": "lu"}
fd.solve(a==L, ud, bcs=[bc], solver_parameters=opts)

# Visualize in paraview
ud.rename = "mySolution"
VTKFile("Solutions/SolPoisson.pvd").write(ud)




firedrake:WARNING OMP_NUM_THREADS is not set or is set to a value greater than 1, we suggest setting OMP_NUM_THREADS=1 to improve performance


That gives us the following surface:

![image.png](imgs/ex_1_plot.png)

* What a mesh is and what geometric entities it contains?

Mesh is the object containing the vertices where the equation is solved and the polygons it forms. In this case, it is square domain with a uniform mesh with triangles as polygons. Each direction is subdivided into 32 rectangles (that are then cut in half to form the triangles).

* What a numerical (discrete) function defined on such a mesh looks like and can be described

A numerical function defined on a mesh can be the value of the function in a finite number of points or a finite set of functions that approximates the original function in each element of the mesh. In the end, for this case, it just looks like a bunch of planes.

* How to describe the finite-dimensional function space that contains these numerical functions.

```python
Vd = fd.FunctionSpace(msh, "CG", degree=1)
```

We are defining our function space based on our mesh and a family of finite-elements. The family used here is Continuous Galerkin. For each vertex there's a function of the base and then the whole function is a linear combination of this base. That's why it depends on the mesh. The degree is the degree of polynomial of the base functions.

A finite-dimensional function space for the numerical solutions is the space in which the solution function lies. In this case, it is the continuous functions that are piecewise C⁰ in each triangle interior 


## Exercise 2

For this exercise, we just need to change the definition of the "mu" variable in the basic code: (run this code after running the previous code block)

In [2]:
import firedrake as fd
from firedrake.output import VTKFile

# Finite element mesh
Nx, Ny = 32, 32
Lx, Ly = 1.0, 1.0
msh = fd.RectangleMesh(Nx, Ny, Lx, Ly, quadrilateral=False)

# Space of functions
Vd = fd.FunctionSpace(msh, "CG", degree=1)

# Test and Trial functions
u  = fd.TrialFunction(Vd)
v  = fd.TestFunction(Vd)

# Boundary conditions
u_boundary = fd.Constant(0.0)
bc = fd.DirichletBC(Vd, u_boundary, "on_boundary")

# Source term
f  = fd.Constant(1.0)

x  = fd.SpatialCoordinate(msh)
mu = 0.01 + fd.exp(-100.0 * ((x[0] - 0.5)**2 + (x[1] - 0.5)**2)) # this is a function of x


# Bilinear form (lhs) and linear form (rhs)
a  = fd.inner(mu * fd.grad(u), fd.grad(v)) * fd.dx
L  = fd.inner(f, v) * fd.dx

# Solve the problem
ud = fd.Function(Vd)
opts={"ksp_type": "preonly", "pc_type": "lu"}
fd.solve(a==L, ud, bcs=[bc], solver_parameters=opts)

# Visualize in paraview
ud.rename = "mySolution"
VTKFile("Solutions/Sol_ex_2.pvd").write(ud)



Plot:

![image-2.png](imgs/ex_2_plot.png)

## Exercise 3

For this exercise, I ran the previous code for the required values of refinement and using or not a quadrilateral mesh.

In [9]:
import firedrake as fd
from firedrake.output import VTKFile

refinement = [16, 32, 64, 128, 512]
quadrilateral = [False, True]

for quad in quadrilateral:
    for ref in refinement:
        # Finite element mesh
        Lx, Ly = 1.0, 1.0
        msh = fd.RectangleMesh(ref, ref, Lx, Ly, quadrilateral=quad)

        # Space of functions
        Vd = fd.FunctionSpace(msh, "CG", degree=1)

        # Test and Trial functions
        u  = fd.TrialFunction(Vd)
        v  = fd.TestFunction(Vd)

        # Boundary conditions
        u_boundary = fd.Constant(0.0)
        bc = fd.DirichletBC(Vd, u_boundary, "on_boundary")

        # Source term
        f  = fd.Constant(1.0)

        x  = fd.SpatialCoordinate(msh)
        mu = 0.01 + fd.exp(- 100.0 * ((x[0] - 0.5)**2 + (x[1] - 0.5)**2)) # this is a function of x!


        # Bilinear form (lhs) and linear form (rhs)
        a  = fd.inner(mu * fd.grad(u), fd.grad(v)) * fd.dx
        L  = fd.inner(f, v) * fd.dx

        # Solve the problem
        ud = fd.Function(Vd)
        opts={"ksp_type": "preonly", "pc_type": "lu"}
        fd.solve(a==L, ud, bcs=[bc], solver_parameters=opts)

        # Visualize in paraview
        ud.rename = "mySolution"
        VTKFile(f"Solutions/Sol_ex_3_ref{ref}_quad_{quad}.pvd").write(ud)

        print(f"Norm L2: ref = {ref}, quad = {quad}: ", fd.norm(ud, "L2"))


Norm L2: ref = 16, quad = False:  3.9123784830960835
Norm L2: ref = 16, quad = True:  3.9420747043314033
Norm L2: ref = 32, quad = False:  3.956028577095724
Norm L2: ref = 32, quad = True:  3.9635799233562006
Norm L2: ref = 64, quad = False:  3.9670040942765694
Norm L2: ref = 64, quad = True:  3.9688995680656705
Norm L2: ref = 128, quad = False:  3.969756139354062
Norm L2: ref = 128, quad = True:  3.970230492238398
Norm L2: ref = 512, quad = False:  3.9706168406509765
Norm L2: ref = 512, quad = True:  3.970646497222527


Triangular mesh:

16x16

![image-5.png](imgs/ex_3_plot_tri_16.png)

128x128

![image-6.png](imgs/ex_3_plot_tri_128.png)

The result gets smoother increasing the refinement


Quadrilateral mesh:

16x16

![image-7.png](imgs/ex_3_plot_quad_16.png)

128x128

![image-8.png](imgs/ex_3_plot_quad_128.png)

It appears that the quadrilateral meshes gives better results when compared with a finer grid, at least in this case.

## Exercise 4

In [4]:
import firedrake as fd
from firedrake.output import VTKFile

# Finite element mesh
Nx, Ny = 32, 32
Lx, Ly = 1.0, 1.0
msh = fd.RectangleMesh(Nx, Ny, Lx, Ly, quadrilateral=False)

# Space of functions
Vd = fd.FunctionSpace(msh, "CG", degree=1)

# Test and Trial functions
u  = fd.TrialFunction(Vd)
v  = fd.TestFunction(Vd)

# Boundary conditions
u_boundary = fd.Constant(0.0)
bc = fd.DirichletBC(Vd, u_boundary, "on_boundary")

# Source term
f  = fd.Constant(1.0)

x  = fd.SpatialCoordinate(msh)
mu = 0.01 + fd.exp(-100.0 * ((x[0] - 0.5)**2 + (x[1] - 0.5)**2)) # this is a function of x


# Bilinear form (lhs) and linear form (rhs)
a  = fd.inner(mu * fd.grad(u), fd.grad(v)) * fd.dx
L  = fd.inner(f, v) * fd.dx

# Solve the problem
ud = fd.Function(Vd)
opts={"ksp_type": "preonly", "pc_type": "lu"}
fd.solve(a==L, ud, bcs=[bc], solver_parameters=opts)

tau = mu * fd.grad(ud)

norm_tau = fd.sqrt(fd.inner(tau, tau))

#Creates the function space 
Vstress = fd.FunctionSpace(msh, "DG", degree=0)

#Creates a function object in this space
stress = fd.Function(Vstress)

# Writes the values of norm_tau in terms of the function basis
stress.interpolate(norm_tau)

stress.rename = "stress_magnitude"
VTKFile("Solutions/stress.pvd").write(stress)

n = fd.FacetNormal(msh)
Wss = fd.inner(tau,n) * fd.ds
print(fd.assemble(Wss))



-0.9384765607532284


Stress plot:

![image.png](imgs/ex_4_plot.png)

## Exercise 5

In [5]:
import gmsh
import firedrake
from math import *

#got this code for the Circle here: https://gitlab.onelab.info/gmsh/gmsh/-/work_items/827
gmsh.initialize()
gmsh.option.setNumber("Mesh.SaveAll", 1)
gmsh.option.setNumber("Mesh.Algorithm", 6)
gmsh.model.add("t1")
lc = 10e-3
gmsh.model.occ.addCircle(0.5, 0.5, 0.0,0.5,1,angle1=0.,angle2=2*pi)
gmsh.model.occ.addCurveLoop([1] ,2)
gmsh.model.occ.addPlaneSurface([2],1)
gmsh.model.addPhysicalGroup(1, [1], 1)
gmsh.model.addPhysicalGroup(2, [1], 2)
gmsh.model.occ.synchronize()
gmsh.model.mesh.generate(2)
gmsh.write("Circle.msh")
gmsh.finalize()

msh = fd.Mesh("Circle.msh")

# Space of functions
Vd = fd.FunctionSpace(msh, "CG", degree=1)

# Test and Trial functions
u  = fd.TrialFunction(Vd)
v  = fd.TestFunction(Vd)

# Boundary conditions
u_boundary = fd.Constant(0.0)
bc = fd.DirichletBC(Vd, u_boundary, "on_boundary")

# Source term
f  = fd.Constant(1.0)

x  = fd.SpatialCoordinate(msh)
mu = 0.01 + fd.exp(-100.0 * ((x[0] - 0.5)**2 + (x[1] - 0.5)**2)) # this is a function of x


# Bilinear form (lhs) and linear form (rhs)
a  = fd.inner(mu * fd.grad(u), fd.grad(v)) * fd.dx
L  = fd.inner(f, v) * fd.dx

# Solve the problem
ud = fd.Function(Vd)
opts={"ksp_type": "preonly", "pc_type": "lu"}
fd.solve(a==L, ud, bcs=[bc], solver_parameters=opts)

# Visualize in paraview
ud.rename = "mySolution"
VTKFile("Solutions/Sol_ex_5.pvd").write(ud)

Info    : Meshing 1D...
Info    : Meshing curve 1 (Circle)
Info    : Done meshing 1D (Wall 0.00103067s, CPU 0.001002s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.0163548s, CPU 0.012716s)
Info    : 71 nodes 141 elements
Info    : Writing 'Circle.msh'...
Info    : Done writing 'Circle.msh'


PETSc Error --- Application was linked against both OpenMPI and MPICH based MPI libraries and will not run correctly


Plot:

![img-ex-5](imgs/ex_5_plot.png)